# Create the main life-cycle inventory database in your brightway project

In [ ]:
import bw2data as bd
import bw2io as bi
from premise import *
from mescal import *

In [ ]:
ei_version = '3.12' # version of the ecoinvent database.
ei_db_name = f"ecoinvent-{ei_version}-cutoff" # name of the ecoinvent database in the brightway project
bd.projects.set_current('pypsa') # set the current brightway project
biosphere_name= 'biosphere3' # name of the biosphere database in the brightway project

### Premise parameters ###
model = 'remind-eu' # IAM used to create the database
pathway = 'SSP2-NDC' # SSP-RCP scenario used to create the database
years = [2020, 2030, 2040, 2050] # years of the database

# Or, define a complete list of scenarios
use_scenarios_list = False
scenarios = [
    {"model": "image", "pathway": "SSP1-L", "year": 2050}, # +1.7°C
    {"model": "image", "pathway": "SSP2-M", "year": 2050}, # +2.9°C
    {"model": "image", "pathway": "SSP3-H", "year": 2050}, # +3.6°C
    {"model": "remind", "pathway": "SSP2-PkBudg1000", "year": 2050}, # +1.8°C
    {"model": "remind", "pathway": "SSP2-NPi", "year": 2050}, # +2.6°C
    {"model": "remind", "pathway": "SSP3-rollBack", "year": 2050}, # +3.5°C
    {"model": "tiam-ucl", "pathway": "SSP2-RCP26", "year": 2050}, # +1.8°C
    {"model": "tiam-ucl", "pathway": "SSP2-RCP45", "year": 2050}, # +2.8°C
    {"model": "tiam-ucl", "pathway": "SSP2-Base", "year": 2050}, # +3.1°C
    {"model": "message", "pathway": "SSP1-L", "year": 2050}, # +1.6°C
    {"model": "message", "pathway": "SSP2-M", "year": 2050}, # +2.9°C
    {"model": "message", "pathway": "SSP3-H", "year": 2050}, # +3.2°C
]

In [ ]:
if ei_version == '3.10.1':
    ei_version_premise = '3.10'
else:
    ei_version_premise = ei_version

## Import ecoinvent

In [ ]:
if f'ecoinvent-{ei_version}-cutoff' in bd.databases:
    print(f'ecoinvent {ei_version} is already present in the project')
else:
    bi.import_ecoinvent_release(
        version=ei_version,
        system_model='cutoff', # can be cutoff / apos / consequential / EN15804
        username='JohnDoe',
        password='1234',
        biosphere_name=biosphere_name,
    )

## Import premise databases

In [ ]:
# Clear cache is encouraged if updating premise or if encountering issues with inventories
clear_cache()

In [ ]:
# Initialize the premise database
ndb = NewDatabase(
    scenarios=scenarios if use_scenarios_list else [{"model": model, "pathway": pathway, "year": year} for year in years],
    source_db=ei_db_name,
    source_version=ei_version_premise,
    key='xxx', # ask a key to Romain Sacchi (romain.sacchi@psi.ch)
    biosphere_name=biosphere_name,
)

In [ ]:
# Update the database with IAM data
ndb.update()

In [ ]:
databases_names = (
    [f'ecoinvent_cutoff_{ei_version}_{model}_{pathway}_{year}' for model, pathway, year in [(s['model'], s['pathway'], s['year']) for s in scenarios]] if use_scenarios_list
    else [f'ecoinvent_cutoff_{ei_version}_{model}_{pathway}_{year}' for year in years]
)

In [ ]:
# Write the database in your brightway project
ndb.write_db_to_brightway(name=databases_names)

## Regionalization of the database

To regionalize the database, clone the [Regiopremise](https://github.com/matthieu-str/Regiopremise) repository and run the [demo.ipynb](https://github.com/matthieu-str/Regiopremise/blob/ei3.10/doc/demo.ipynb) notebook. 

## Archiving, Restoring, and Sharing Projects

In [ ]:
bi.backup.backup_project_directory(
    project=f'ecoinvent{ei_version}',
)

In [ ]:
bi.backup.restore_project_directory(
    fp='./export/backup/file_name.tar.gz',
    project_name=f'ecoinvent{ei_version}',
    overwrite_existing=True,
)